# IMPORTATION DES PACKAGES

In [19]:
import os
import time
import requests
import pandas as pd

In [ ]:
HEADERS = {"User-Agent": "ProjetPython 1"}

In [29]:
dernier_appel = 0

def lire(chemin, params):
    global dernier_appel
    params["fmt"] = "json"
    for essai in range(4):
        attente = 1.1 - (time.time() - dernier_appel)
        if attente > 0:
            time.sleep(attente)
        dernier_appel = time.time()
        try:
            r = requests.get("https://musicbrainz.org/ws/2/" + chemin, headers=HEADERS, params=params)
            if r.status_code == 200:
                return r.json()
            print("Refus, code", r.status_code)
        except requests.RequestException:
            print("Pas de connexion")
        time.sleep(15 * (essai + 1))      # 15 s, puis 30 s, puis 45 s...
    return None                           # None = échec réel, à distinguer d'un résultat vide


def nettoyer_titre(titre):
    for sep in [" (", " [", " - "]:
        if titre.find(sep) > 0:
            titre = titre[:titre.find(sep)]
    return titre.strip().replace('"', '')


def langue_musique(artiste, titre):
    noms = [n.strip().lower().replace('"', '') for n in artiste.split(",")]
    minimum = 1 if len(noms) == 1 else 2
    requete = f'recording:"{nettoyer_titre(titre)}" AND (' + " OR ".join(f'artist:"{n}"' for n in noms) + ")"

    # Requête 1 : la recherche
    resultat = lire("recording", {"query": requete, "limit": 10})
    if resultat is None:
        return "ERREUR"

    # Garder la première version dont les artistes correspondent (aucune requête ici)
    for version in resultat.get("recordings", []):
        communs = sum(1 for c in version.get("artist-credit", []) if c.get("name", "").lower() in noms)
        if communs < minimum:
            continue

        # Requête 2 : la fiche de cette version, avec ses oeuvres ET ses sorties
        fiche = lire("recording/" + version["id"], {"inc": "work-rels+releases"})
        if fiche is None:
            return "ERREUR"

        # Langue de l'oeuvre, directement dans le lien (pas de requête en plus)
        for relation in fiche.get("relations", []):
            langues = (relation.get("work") or {}).get("languages")
            if langues:
                return ";".join(langues)

        # Sinon, langue de l'album ou du single
        for sortie in fiche.get("releases", []):
            code = (sortie.get("text-representation") or {}).get("language")
            if code:
                return code

        return ""        # version trouvée, mais aucune langue renseignée

    return ""            # aucune version ne correspond

In [30]:
print(langue_musique("Stromae", "Alors on danse"))

Refus, code 503
eng


In [31]:
print(langue_musique("DJ Snake, Selena Gomez, Ozuna, Cardi B", "Taki Taki (with Selena Gomez, Ozuna & Cardi B)"))

Refus, code 503
spa;eng


In [32]:
NUMERO = 4          # 1, 2, 3 ou 4 selon le notebook

CHEMIN_CHANSONS = "chansons.csv"
CHEMIN_TRANCHE = f"chansons_{NUMERO}.csv"

# Reprendre le fichier de la tranche s'il existe déjà, sinon partir de la liste complète
if os.path.exists(CHEMIN_TRANCHE):
    chansons = pd.read_csv(CHEMIN_TRANCHE, keep_default_na=False)
else:
    chansons = pd.read_csv(CHEMIN_CHANSONS, keep_default_na=False)

# Limites de la tranche
n = len(chansons)
debut = (NUMERO - 1) * n // 4
fin = NUMERO * n // 4
print("Tranche", NUMERO, ": lignes", debut, "à", fin - 1)

for i in range(debut, fin):
    if chansons.loc[i, "langue"] != "":
        continue

    chansons.loc[i, "langue"] = langue_musique(chansons.loc[i, "artist_names"],
                                               chansons.loc[i, "track_name"])

    if i % 20 == 0:
        chansons.to_csv(CHEMIN_TRANCHE, index=False)
        print(i, "/", fin)

chansons.to_csv(CHEMIN_TRANCHE, index=False)
print("Tranche", NUMERO, "terminée !")

Tranche 1 : lignes 0 à 5097
0 / 5098
Refus, code 503
Refus, code 503
Refus, code 503
Refus, code 503
Refus, code 503
Refus, code 503
Refus, code 503
Refus, code 503
Refus, code 503


KeyboardInterrupt: 